# 31 · 加进 Orphanet 的病：同一个病，两本目录

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → 28 时间外验证 ✅ → 29 外部验证 ✅ → 30 离线试打分 ✅ → [31 加进 Orphanet 的病 ✅] → 32 新架构
```

> **这一章的代码在分支 `feat/orphanet-diseases` 里。运行这本笔记不花钱**：不调用任何大模型。约 1 分钟。

## 背景：知识库只读了一半

罕见病有两本最常用的"目录"，各自给每个病编一个号：

| 目录 | 是什么 | 编号长什么样 |
|---|---|---|
| **OMIM** | 美国约翰霍普金斯大学维护，以**遗传病**为主，按致病基因分得很细 | `OMIM:109650` |
| **Orphanet** | 欧洲的罕见病目录，**所有罕见病**都收，包括不是单基因遗传的（很多免疫病） | `ORPHA:117` |

HPO（症状词典，第 10 章）给两本目录里的病**都**标了症状。可我们的知识库从第 10 章起，读文件时只留 `OMIM:` 开头的行（[`knowledge/hpo.py`](../../src/raredx/knowledge/hpo.py)），Orphanet 的 4,355 种病、11.7 万行症状标注全丢了。

后果在第 29、30 章都看到了：

- HMS 门诊的 88 个病人里，**24 个的病只在 Orphanet 里有**（比如抗合成酶综合征，一种肌肉和肺都发炎的免疫病），任何用我们知识库的系统都答不对
- **白塞病**（反复口腔溃疡、眼睛发炎的免疫病）在 OMIM 那边只记了 15 条症状，Orphanet 那边记了 85 条。只按 OMIM 的 15 条算"像不像"，它比不过一个记得更全的罕见病

**难点不在"多读几行"，而在"同一个病有两个编号"**：白塞病在 OMIM 是 `OMIM:109650`，在 Orphanet 是 `ORPHA:117`。如果当成两个病，名单里会出现两个白塞病，判分也会乱。

**公司里的做法**：这叫**实体对齐**（entity resolution）或"主数据管理"：两个系统给同一个东西编了不同的号，要有一张权威的对照表。医疗数据里，疾病编号的权威对照表是 **Mondo 疾病本体**（Monarch Initiative 维护，ClinGen、GA4GH 的标准都用它）。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 拿到"哪两个编号是同一个病"的对照表 | Mondo 的 SSSOM 对照文件，固定版本 | 同一个病不再算两次 |
| 2 | 知识库读进 Orphanet，合并同一个病 | 按 Mondo 合并；对不上 OMIM 的单独成病 | 白塞病 15 条 → 93 条；抗合成酶综合征有了 |
| 3 | 防泄露：新规则 + 检查 | 遮挡按 OMIM 那部分判；`raredx eval coverage` | Orphanet 不写出处，不能让它把答案带进来 |
| 4 | 编号和判分：OMIM、ORPHA 都认 | 工具、判分、提示词一起改 | 系统能答 Orphanet 的病，判分认对等编号 |
| 5 | 结果：不用模型的系统，换了知识库以后 | 四个开发来源，逐题配对 | 先看知识本身带来多少 |

**花费**：0。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print("\n".join(line for line in out.splitlines() if "INFO mlflow" not in line))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 对照表：Mondo 说哪两个编号是同一个病

### 在干什么
下载 Mondo 的对照文件，固定版本，用它判断"一个 OMIM 编号和一个 ORPHA 编号是不是同一个病"。

### 背景
Orphanet 自己其实也有一张"对应到 OMIM"的表，项目里算患病率时已经在用。可它**不全**：比如红斑狼疮 `ORPHA:536`，Orphanet 只把它对到两个"易感基因位点"（OMIM 把红斑狼疮按易感基因分成十几个条目），偏偏没有对到红斑狼疮本身 `OMIM:152700`。只用 Orphanet 的表，知识库里就会有两个红斑狼疮。

### 用了什么技术，为什么

| 选择 | 为什么 |
|---|---|
| **Mondo 疾病本体** | 专门做"各家疾病目录对齐"的开放标准，Monarch Initiative 维护，ClinGen 等都用。它给每个病一个 Mondo 编号，列出各家目录里**完全等同**的编号 |
| **SSSOM 文件**（`mondo.sssom.tsv`） | 对照关系的标准表格格式：一行一对，写明是"完全等同"（exactMatch）还是"更宽 / 更窄"。我们**只认完全等同** |
| 版本 `v2026-09-01` | 和 HPO 的版本同一天；登记在 `evals/datasets.toml`，带校验码，和其他数据一个规矩 |

**打个比方**：两家公司合并，员工在两家各有一个工号。HR 有一张"两个工号是同一个人"的对照表，Mondo 就是这张表。

### 公司里什么情况怎么做
- 对齐两个系统的编号，**用权威的对照表，不自己凭名字猜**（名字相似的病可能不是同一个，比如各种"红斑狼疮 1 型、2 型"）
- 对照表本身也是数据，**固定版本、记校验码**

### 执行

In [ ]:
run("uv run raredx data fetch mondo")
from raredx.config import settings
from raredx.data import sources
from raredx.knowledge.diseases import MAPPINGS, Identity
registry = sources.load_sources(settings.evals_dir / "datasets.toml")
identity = Identity(sources.raw_dir(registry["mondo"], settings.data_dir) / MAPPINGS)
for disease in ("OMIM:109650", "OMIM:152700", "ORPHA:81"):
    print(f"{disease:<13} Mondo {identity.mondo(disease)}  同一个病：{sorted(identity.same(disease))}")

### 结果怎么看
- 第一行：文件已经在本机、校验码对得上（重复运行安全）
- **白塞病** `OMIM:109650` 和 `ORPHA:117` 是同一个 Mondo 病
- **红斑狼疮** `OMIM:152700` 和 `ORPHA:536` 也是同一个：Orphanet 自己的表漏掉的，Mondo 补上了
- **抗合成酶综合征** `ORPHA:81`：没有对应的 OMIM 编号（它不是单基因遗传病）

### 面试怎么说
> "HPO annotates diseases under both OMIM and Orphanet ids. I used Mondo's SSSOM mappings — exact matches only, pinned to the release dated like HPO's — as the single source of truth for which ids name the same disease. Orphanet's own mapping missed lupus; Mondo has it."

---

## 第 2 步 · 知识库读进 Orphanet，合并同一个病

### 在干什么
知识库加载时，也读 Orphanet 的症状标注，按 Mondo 合并。

### 背景
Orphanet 的 4,355 种病，按 Mondo 分三类：

| 情况 | 怎么处理 | 例子 |
|---|---|---|
| Mondo 说它**恰好等于一个** OMIM 的病（2,122 种） | 它的症状并进那个 OMIM 病，用 OMIM 的编号和名字；它的 ORPHA 编号记成"别名" | 白塞病：`ORPHA:117` 并进 `OMIM:109650` |
| 没有对应的 OMIM 病（2,231 种） | 自己成一个病，用 ORPHA 编号 | 抗合成酶综合征 `ORPHA:81` |
| 对应**好几个** OMIM 病（2 种） | 拿不准，单独成病，不合并 | — |

### 用了什么技术，为什么
- **以 OMIM 编号为主**：之前所有系统、考题、记录都用 OMIM 编号。合并以后旧的编号全都还有效，只是背后的症状多了
- **"别名"**：工具收到 `ORPHA:117`，自动当成 `OMIM:109650`。模型写哪个编号都能查到
- 一个新函数 `canonical()`：任何 OMIM 或 ORPHA 编号 → 知识库里它指的那个病

### 公司里什么情况怎么做
- 合并两份数据时**保留一个主编号**，另一个当别名，老的引用都不会断
- 合并的规则写死在代码里、有测试：哪种情况合、哪种不合

### 执行
同一个 HPO 版本，读两次：一次只读 OMIM（和以前一样），一次读进 Orphanet：

In [ ]:
from raredx.knowledge import hpo
folder = sources.raw_dir(registry["hpo"], settings.data_dir)
before, after = hpo.load(folder), hpo.load(folder, identity)
print("以前：", before)
print("现在：", after)

def n_features(knowledge, disease):
    here = knowledge.canonical(disease)
    return len([f for f in knowledge.features(here) if not f.absent]) if here else None

print(f"\n{'':<34}{'以前记的症状':>10}{'现在':>8}   现在叫")
for disease, name in (("OMIM:109650", "白塞病"), ("OMIM:152700", "系统性红斑狼疮"), ("OMIM:608710", "肉芽肿性多血管炎"),
                      ("ORPHA:81", "抗合成酶综合征"), ("ORPHA:761", "IgA 血管炎"), ("ORPHA:117", "白塞病（Orphanet 编号）")):
    print(f"{name:<26}{str(n_features(before, disease) or '没有'):>12}{n_features(after, disease):>8}   {after.canonical(disease)}")

### 结果怎么看
- 第一行"以前"：只有 OMIM 的病。第二行"现在"：OMIM 的病里并进了 2,122 个 Orphanet 条目，另外多了 2,233 个 Orphanet 自己的病
- **白塞病**的症状从 15 条变成 93 条，**红斑狼疮** 21 → 52，**肉芽肿性多血管炎** 32 → 95
- **抗合成酶综合征**、**IgA 血管炎**以前在知识库里根本没有，现在有了
- 最后一行：用 Orphanet 的编号 `ORPHA:117` 去查，得到的是 `OMIM:109650`，同一个病

### 面试怎么说
> "The knowledge now merges each Orphanet entry Mondo equates with exactly one OMIM disease into it, keeping the OMIM id so every earlier reference holds, and keeps the rest as diseases of their own. Behçet disease went from 15 annotated features to 93; antisynthetase syndrome, which no system could name before, is now a disease the tools rank."

---

## 第 3 步 · 防泄露：Orphanet 不写出处，怎么遮

### 在干什么
定一条新的遮挡规则，再检查它有没有漏。

### 背景
第 21 章起，每道题都会**遮住它自己的出处论文**：知识库里只来自这篇论文的标注，答这道题时当作不存在。OMIM 的每一行标注都写着出自哪篇论文（`PMID:…`），所以能遮。

**Orphanet 的标注不写论文**，出处一栏写的是它自己的条目号（`ORPHA:117`），没法按论文遮。危险在这里：一个只在一篇论文里报道过的病，Orphanet 给它写的介绍，很可能就是照这篇论文写的。如果遮了 OMIM 那部分、留着 Orphanet 那部分，答案就从侧门进来了。

### 用了什么技术，为什么
**新规则**：判断"这个病是不是只来自被遮的论文"时，**只看 OMIM 那部分**。如果是，这个病**整个藏起来**，并进来的 Orphanet 标注一起藏。写给 LIRICAL 的标注文件也照这条规则删。

**检查**：用一个新命令 `raredx eval coverage`，比一比加 Orphanet 前后，每份题里"系统有可能答对"的题数：
- 开发用的 6,014 个 Phenopacket 病人：如果加了 Orphanet 以后，有题从"遮挡后答不了"变成"答得了"，就说明有漏，必须一道道查
- 只查开发用的数据。留出集和考卷不碰（命令会拒绝）

### 公司里什么情况怎么做
- **新加一个数据来源，先查它会不会把测试答案带进来**（data leakage）。这里查的是"可答题数"，只看知识、不看任何系统的回答，所以不算偷看考卷
- 不能遮的东西，就**按最保守的方式处理**：宁可多藏一点

### 执行

In [ ]:
run("uv run raredx eval coverage --split design --split dev-200 --split rarebench-hms --split rarebench-ramedis --split rarebench-mme --show 6")

### 结果怎么看
- `design`（开发用的全部 6,014 个病人）：**5,004 → 5,004**，一道都没多。新规则没有让任何被遮的病从 Orphanet 那边漏进来，**泄露检查通过**
- `dev-200` 159 → 159，RAMEDIS 624、MME 40 都不变
- **HMS：64 → 85**。那 24 道"注定答不对"的题，21 道现在**有可能**答对了。`+` 开头的几行就是它们：IgA 血管炎、抗合成酶综合征、木村病……都是 Orphanet 才有的病
- 注意这只是"有可能"：病在知识库里了，系统还得把它找出来、排到第一。下一步看真的答对了多少

### 面试怎么说
> "Orphanet's annotations cite no paper, so they can't be masked per case. I made the rule conservative — a disease whose OMIM annotation rests only on the masked paper is hidden whole, Orphanet's lines with it — and checked it: across 6,014 development cases not one became answerable that wasn't before, while on the clinic set 21 of 24 previously impossible cases became answerable."

---

## 第 4 步 · 编号和判分：OMIM、ORPHA 都认

### 在干什么
让整个系统都接受 Orphanet 的编号。

### 背景
从第 07 章起，"答对"的规则是"系统答的 OMIM 编号等于确诊的 OMIM 编号"。现在要改三处：

| 哪里 | 以前 | 现在 |
|---|---|---|
| 工具（排序、核对、查病） | 只收 `OMIM:` 编号 | 也收 `ORPHA:`；别名自动换成它并进去的那个病 |
| 判分 | 只认 OMIM 编号 | 也认 ORPHA 编号；**Mondo 说是同一个病的编号都算对**（答 `ORPHA:117`，确诊写 `OMIM:109650`，算对）。每次考试都记下用了哪个版本的 Mondo |
| 提示词 | "病人疑似患有**罕见单基因遗传病**"；答案只能写 OMIM 编号 | "病人疑似患有**罕见病**"；没有 OMIM 编号的病写 ORPHA 编号 |

提示词改了，**版本号都往上加一**：直接猜 `direct-v2`、固定流程 `pipeline-v2`、三种 agent `agent-v4`、`seeded-v2`、`hypotheses-v3/v4`。这样新旧结果永远不会混在一起。

### 用了什么技术，为什么
- 判分的"同一个病"规则做成一个**可以传进去的规则**，而不是改题目的答案：MLflow 把每份题的标准答案存过一次就复用，改答案会让新旧考试用的答案不一样
- 老的考试记录可以用新规则**重新判分**（`raredx eval score`），不用重新问模型。这样新旧系统在同一把尺子下比

### 公司里什么情况怎么做
- **改评分规则，就把所有要比较的结果都按新规则重判一遍**，不然比的是两把不同的尺子
- 提示词是代码的一部分：改了就升版本号，跟着每次运行记下来

### 执行
这一步是代码改动，有测试覆盖（`tests/knowledge/test_orphanet_merged.py` 等），这里只演示判分规则：

In [ ]:
from raredx.evals import scoring
from raredx.evals.systems import Candidate
answer = [Candidate(disease_name="Behçet disease", disease_id="ORPHA:117")]
print("只按写下的编号：", scoring.ranks(answer, "OMIM:109650", None))
print("认 Mondo 的对等：", scoring.ranks(answer, "OMIM:109650", None, [], identity.same))

### 结果怎么看
- 括号里第一个数是"答在第几名"，`None` 是没答到
- 只按写下的编号：`ORPHA:117` ≠ `OMIM:109650`，算没答到
- 认 Mondo 的对等：是同一个病，第 1 名答对

### 面试怎么说
> "Scoring takes a disease-identity rule rather than rewritten answer keys, because MLflow reuses each manifest's registered keys; old runs are rescored under the same rule from their recorded answers, so before and after are measured with one ruler. Every prompt that changed got a new version."

---

## 第 5 步 · 结果：知识本身带来了多少

### 在干什么
先不用模型，只看**两个不用模型的系统**（第 10 章的按症状排序、第 14 章的先排名再逐条核对）换了知识库以后，在四个开发来源上答对多少。

### 背景
这样能把"知识的变化"和"模型的变化"分开：不用模型的系统，分数变了只能是知识库变了。用模型的系统（直接猜、agent）因为提示词也改了，留到下一章一起量。

为了公平，旧的考试记录先按新的判分规则**重判**一遍，再和新考试逐题比。

### 用了什么技术，为什么
- 新考试：`raredx eval run`，代码版本 `73e2e67`，不花钱
- 旧考试重判：`raredx eval score`
- 逐题比：`raredx eval hits` 把新旧两次放进一张表

### 执行

In [ ]:
import csv
from raredx.evals.scoring import p_value, sign_test

def load(name):
    return list(csv.DictReader(open(ROOT / "docs" / name, encoding="utf-8"), delimiter="\t"))

SOURCES = {"dev-200": "dev-200", "rarebench-hms": "HMS", "rarebench-ramedis": "RAMEDIS", "rarebench-mme": "MME"}
for k in (1, 10):
    print(f"\n{'第 1 名' if k == 1 else '前 10 名':<26}{'以前':>8}{'现在':>8}   逐题（只有现在对 : 只有以前对）")
    for split, label in SOURCES.items():
        rows = load(f"orphanet_hits_{split}.tsv")
        for system, name in (("phenotype-rank", "按症状排序"), ("rank-then-check", "先排名再逐条核对")):
            old, new = f"{system}-omim@{k}", f"{system}-orphanet@{k}"
            if old not in rows[0]:
                continue
            before = sum(r[old] == "1" for r in rows) / len(rows)
            after = sum(r[new] == "1" for r in rows) / len(rows)
            w = sum(r[new] == "1" and r[old] == "0" for r in rows)
            l = sum(r[old] == "1" and r[new] == "0" for r in rows)
            print(f"{label:<8}{name:<16}{before:>9.1%}{after:>8.1%}   {w} : {l}  {p_value(sign_test(w, l))}")

### 结果怎么看
有涨有跌：

- **HMS 涨了**：先排名再逐条核对的前 10 名从 33.0% 到 **46.6%**（18 : 6，p = 0.023，显著）。第 1 名 6.8% → 8.0%，差别不大
- **RAMEDIS 掉了，而且显著**：第 1 名 11.9% → **6.7%**（17 : 49），前 10 名 58.0% → **43.1%**（30 : 123）
- **dev-200 往下**：第 1 名 30.5% → 27.0%（3 : 10，p = 0.092，没到显著）
- MME 差不多；按症状排序这个系统，各来源变化都不大

按我们定的规矩（哪个来源都不许显著掉），**这一版知识库不能直接收**。下一步找原因。

### 面试怎么说
> "Before any model run I measured the knowledge change alone, on the no-model systems: on the clinic set the top ten rose from 33% to 47%, but on the metabolic registry it fell from 58% to 43%. By the rule that no source may drop significantly, it could not go in as it was."

---

## 第 6 步 · 为什么有涨有跌，接下来怎么办

### 在干什么
找出 RAMEDIS 掉分的原因，试几种改法，定下一步。

### 背景：原因出在"像不像"的算法上
RAMEDIS 里从前 10 名掉出去的 123 道题，**63 道是苯丙酮尿症**（一种天生缺酶、吃进去的苯丙氨酸代谢不掉的病）。苯丙酮尿症在 OMIM 那边记了 27 条症状，合并 Orphanet 以后变成 **46 条**。

症状排序（第 10 章）算"像不像"是**两个方向取平均**：
1. 病人的每个症状，在这个病里找最像的一条，看对得上多少
2. 这个病的每条症状，在病人身上找最像的一个，看对得上多少

简化地算一下：一个 RAMEDIS 病人一般只有 9 个左右的症状。假设都是苯丙酮尿症的症状——
- 方向 1：9 个都对得上，两次都是满分
- 方向 2：病有 27 条时，病人身上对得上的约 9 / 27 ≈ 三分之一；变成 46 条以后，只有 9 / 46 ≈ 五分之一

**病记得越全，方向 2 的分越低。** 症状少的病人（RAMEDIS）吃亏最大；症状多的病人（HMS 的门诊病人有 18 到 43 个症状）反而沾光。另外，新加的 2,233 个 Orphanet 自己的病也会挤进候选名单。

### 试过的改法（在开发数据上用一个临时脚本算的，探索性）

| 先排名再逐条核对，第 1 / 前 10 名 | dev-200 | HMS | RAMEDIS | MME |
|---|---|---|---|---|
| 以前（只有 OMIM） | 30.5 / 54.5 | 6.8 / 33.0 | 11.9 / 58.0 | 60.0 / 85.0 |
| A 合并，两个方向取平均（这一章的版本） | 27.0 / 53.5 | 8.0 / **46.6** | 6.7 / 43.1 | 57.5 / 87.5 |
| B 合并，两个方向取大的 | 28.5 / 44.5 | 5.7 / 23.9 | 7.7 / 51.8 | 50.0 / 77.5 |
| C 合并，另一种算法（BMA） | 20.5 / 35.0 | 6.8 / 29.5 | 5.4 / 16.2 | 52.5 / 65.0 |
| D 排序只用 OMIM 的症状，核对才用合并的 | 30.0 / 52.5 | 8.0 / 35.2 | 10.6 / 46.6 | 62.5 / 85.0 |
| E 不合并，只加 Orphanet 自己的病 | 29.5 / 53.0 | 3.4 / 27.3 | 11.5 / 45.4 | 57.5 / 85.0 |

**没有一种既保住 HMS 的提升、又不伤 RAMEDIS。**

### 定下来的做法：做成开关，用真正的 agent 来定
- 合并 Orphanet 做成一个**开关**：`raredx eval run --orphanet`，或者设置 `RAREDX_ORPHANET=1`。**默认关**：不打开时，知识库和以前一模一样，不会有任何来源掉分
- 每次考试都记下用的是哪种知识库（`knowledge` 一项）
- 不受开关影响、已经生效的：编号和判分认 Orphanet、Mondo 的"同一个病"规则、提示词改成"罕见病"、患病率经 Mondo 查得更全
- **为什么要用 agent 来定**：上面都是不用模型的系统。agent 最后由模型判断，模型自己知道苯丙酮尿症长什么样，不一定跟着掉；HMS 上多出来的病，也要模型去认。规矩里说的"哪个来源都不许掉"，看的是整个系统

### 公司里什么情况怎么做
- 一个改动在一部分用户身上好、另一部分坏，**先做成开关（feature flag）**，默认关，拿完整系统的数据来决定开不开
- 找原因要落到**具体的病、具体的数字**（这里是苯丙酮尿症 27 → 46 条），而不是停在"整体掉了"

---

## 本章小结

| 步骤 | 用的技术 | 在公司里的作用 |
|---|---|---|
| 1 对照表 | Mondo 的 SSSOM 文件，只认完全等同，固定版本 | 实体对齐用权威对照表 |
| 2 合并 | 以 OMIM 编号为主，ORPHA 当别名；对不上的单独成病 | 合并数据不断老引用 |
| 3 防泄露 | 遮挡按 OMIM 那部分判，整病藏起；`raredx eval coverage` 检查 | 新数据来源先查泄露 |
| 4 编号和判分 | 工具、判分、提示词都认 ORPHA；判分认 Mondo 的对等；提示词升版本 | 改尺子就把旧结果按新尺子重判 |
| 5 结果 | 不用模型的系统，四个来源逐题比 | 先单独量知识的变化 |
| 6 原因和下一步 | 找到苯丙酮尿症 27 → 46 条；试 5 种改法；做成开关，默认关 | 有好有坏的改动，用开关和完整系统的数据来定 |

**一句话**：用 Mondo 把 Orphanet 的 4,355 种病接进知识库，防泄露检查通过，HMS 上 21 道原来注定答不对的题有了希望；但"像不像"的算法对症状表长的病不利，RAMEDIS 显著掉分，所以先做成默认关的开关，下一章用 agent 实测再定。

## 面试题

> **Q: You added a second disease catalogue and one dataset got worse. What did you do?**
>
> A: "Found the mechanism before touching anything: the phenotype similarity averages both directions, so a disease whose annotation grew from 27 to 46 features scored lower for patients with nine findings — the metabolic registry — while patients with many findings gained. I tried five variants; none was free of the loss. So the merge is a flag, off by default, to be decided on the full agent's results, not the no-model ranker's."

> **Q: How do you keep a knowledge source that can't be masked from leaking test answers?**
>
> A: "Make the rule conservative and check it. Orphanet cites no papers, so a disease whose OMIM annotation rests only on the masked paper is hidden whole, Orphanet's lines included. Then I counted answerable cases with and without Orphanet over 6,014 development cases: not one changed."

## 下一章

**32 · 用 agent 定开关**：在四个开发来源上，用 v1.1 的 agent（提示词已改成"罕见病"、认 Orphanet 编号）分别在开关开和关的知识库上考一次，再加上新提示词的"直接猜"当参照。会花钱：先在 ci-20 上量单价，再定上限。